In [ ]:
1. Load engineered datasets
        ↓
2. Inspect model-ready data
        ↓
3. Chronological train/validation/test split
        ↓
4. Create Persistence predictions
        ↓
5. Calculate MAE
        ↓
6. Calculate RMSE
        ↓
7. Calculate R²
        ↓
8. Save baseline predictions/results

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
plant1 = pd.read_csv("../data/final/plant1_features.csv")
plant2 = pd.read_csv("../data/final/plant2_features.csv")

plant1["DATE_TIME"] = pd.to_datetime(plant1["DATE_TIME"])
plant2["DATE_TIME"] = pd.to_datetime(plant2["DATE_TIME"])

print("Plant 1 shape:", plant1.shape)
print("Plant 2 shape:", plant2.shape)

Plant 1 shape: (66640, 25)
Plant 2 shape: (65564, 25)


In [3]:
print("Plant 1:")
display(plant1.head())

print("\nPlant 2:")
display(plant2.head())

Plant 1:


,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,...,month,hour_sin,hour_cos,AC_POWER_lag_1,AC_POWER_lag_4,AC_POWER_lag_8,AC_POWER_lag_96,AC_POWER_rolling_mean_4,AC_POWER_rolling_mean_8,TARGET_AC_POWER
0,2020-05-16 02:45:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6265313.0,21.496287,19.624870,0.0,...,5,0.659346,0.751840,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2020-05-16 03:00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6265313.0,21.645913,20.065206,0.0,...,5,0.707107,0.707107,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2020-05-16 03:15:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6265313.0,21.674314,20.112160,0.0,...,5,0.751840,0.659346,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2020-05-16 03:30:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6265313.0,21.731421,20.351884,0.0,...,5,0.793353,0.608761,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2020-05-16 03:45:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6265313.0,21.780166,20.505894,0.0,...,5,0.831470,0.555570,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Plant 2:


,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,...,month,hour_sin,hour_cos,AC_POWER_lag_1,AC_POWER_lag_4,AC_POWER_lag_8,AC_POWER_lag_96,AC_POWER_rolling_mean_4,AC_POWER_rolling_mean_8,TARGET_AC_POWER
0,2020-05-16 00:15:00,4136001,4UPUqMRk7TRMgml,0.0,0.0,4201.0,2433212.0,26.687313,24.529142,0.0,...,5,0.065403,0.997859,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2020-05-16 00:30:00,4136001,4UPUqMRk7TRMgml,0.0,0.0,4201.0,2433212.0,26.633028,24.688451,0.0,...,5,0.130526,0.991445,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2020-05-16 00:45:00,4136001,4UPUqMRk7TRMgml,0.0,0.0,4201.0,2433212.0,26.676576,24.880440,0.0,...,5,0.195090,0.980785,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2020-05-16 01:00:00,4136001,4UPUqMRk7TRMgml,0.0,0.0,3360.8,2433212.0,26.440456,24.553809,0.0,...,5,0.258819,0.965926,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2020-05-16 01:15:00,4136001,4UPUqMRk7TRMgml,0.0,0.0,0.0,2433212.0,26.193042,24.117222,0.0,...,5,0.321439,0.946930,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
TARGET = "TARGET_AC_POWER"

print("Target:", TARGET)

print("\nPlant 1 target statistics:")
print(plant1[TARGET].describe())

print("\nPlant 2 target statistics:")
print(plant2[TARGET].describe())

Target: TARGET_AC_POWER

Plant 1 target statistics:
count    66640.000000
mean       309.351568
std        396.184925
min          0.000000
25%          0.000000
50%         42.245536
75%        626.542411
max       1410.950000
Name: TARGET_AC_POWER, dtype: float64

Plant 2 target statistics:
count    65564.000000
mean       239.195114
std        359.179861
min          0.000000
25%          0.000000
50%          0.000000
75%        433.168333
max       1385.420000
Name: TARGET_AC_POWER, dtype: float64


In [5]:
print("Plant 1 time range:")
print("Start:", plant1["DATE_TIME"].min())
print("End:  ", plant1["DATE_TIME"].max())

print("\nPlant 2 time range:")
print("Start:", plant2["DATE_TIME"].min())
print("End:  ", plant2["DATE_TIME"].max())

Plant 1 time range:
Start: 2020-05-16 02:45:00
End:   2020-06-17 23:30:00

Plant 2 time range:
Start: 2020-05-16 00:15:00
End:   2020-06-17 23:30:00


In [6]:
def chronological_split(df, train_ratio=0.70, val_ratio=0.15):
    """
    Split a time-series dataframe using timestamp cutoffs.
    """

    df = df.sort_values("DATE_TIME").copy()

    unique_times = np.sort(df["DATE_TIME"].unique())

    n_times = len(unique_times)

    train_end = int(n_times * train_ratio)
    val_end = int(n_times * (train_ratio + val_ratio))

    train_end_time = unique_times[train_end]
    val_end_time = unique_times[val_end]

    train = df[df["DATE_TIME"] < train_end_time].copy()

    validation = df[
        (df["DATE_TIME"] >= train_end_time) &
        (df["DATE_TIME"] < val_end_time)
    ].copy()

    test = df[df["DATE_TIME"] >= val_end_time].copy()

    return train, validation, test

In [7]:
plant1_train, plant1_val, plant1_test = chronological_split(plant1)

plant2_train, plant2_val, plant2_test = chronological_split(plant2)

print("Plant 1:")
print("Train:", plant1_train.shape)
print("Validation:", plant1_val.shape)
print("Test:", plant1_test.shape)

print("\nPlant 2:")
print("Train:", plant2_train.shape)
print("Validation:", plant2_val.shape)
print("Test:", plant2_test.shape)

Plant 1:
Train: (46444, 25)
Validation: (10098, 25)
Test: (10098, 25)

Plant 2:
Train: (44686, 25)
Validation: (10428, 25)
Test: (10450, 25)


In [8]:
print("PLANT 1")
print("Train:", plant1_train["DATE_TIME"].min(), "→", plant1_train["DATE_TIME"].max())
print("Validation:", plant1_val["DATE_TIME"].min(), "→", plant1_val["DATE_TIME"].max())
print("Test:", plant1_test["DATE_TIME"].min(), "→", plant1_test["DATE_TIME"].max())

print("\nPLANT 2")
print("Train:", plant2_train["DATE_TIME"].min(), "→", plant2_train["DATE_TIME"].max())
print("Validation:", plant2_val["DATE_TIME"].min(), "→", plant2_val["DATE_TIME"].max())
print("Test:", plant2_test["DATE_TIME"].min(), "→", plant2_test["DATE_TIME"].max())

PLANT 1
Train: 2020-05-16 02:45:00 → 2020-06-08 09:30:00
Validation: 2020-06-08 09:45:00 → 2020-06-13 04:15:00
Test: 2020-06-13 04:30:00 → 2020-06-17 23:30:00

PLANT 2
Train: 2020-05-16 00:15:00 → 2020-06-08 02:15:00
Validation: 2020-06-08 02:30:00 → 2020-06-13 00:45:00
Test: 2020-06-13 01:00:00 → 2020-06-17 23:30:00


In [9]:
# Persistence baseline
# Predict next 15-minute AC power using current AC power

plant1_test = plant1_test.copy()
plant2_test = plant2_test.copy()

plant1_test["PERSISTENCE_PREDICTION"] = plant1_test["AC_POWER"]
plant2_test["PERSISTENCE_PREDICTION"] = plant2_test["AC_POWER"]

print("Plant 1 persistence predictions created.")
print("Plant 2 persistence predictions created.")

Plant 1 persistence predictions created.
Plant 2 persistence predictions created.


In [10]:
print("Plant 1:")
display(
    plant1_test[
        ["DATE_TIME", "SOURCE_KEY", "AC_POWER",
         "TARGET_AC_POWER", "PERSISTENCE_PREDICTION"]
    ].head(10)
)

print("\nPlant 2:")
display(
    plant2_test[
        ["DATE_TIME", "SOURCE_KEY", "AC_POWER",
         "TARGET_AC_POWER", "PERSISTENCE_PREDICTION"]
    ].head(10)
)

Plant 1:


,DATE_TIME,SOURCE_KEY,AC_POWER,TARGET_AC_POWER,PERSISTENCE_PREDICTION
8640,2020-06-13 04:30:00,3PZuoBAID5Wc2HD,0.0,0.0,0.0
23767,2020-06-13 04:30:00,YxYtjZvoooNbGkE,0.0,0.0,0.0
2598,2020-06-13 04:30:00,1BY6WEcLGh8j5v7,0.0,0.0,0.0
38933,2020-06-13 04:30:00,iCRJl6heRkivqQ3,0.0,0.0,0.0
48021,2020-06-13 04:30:00,rGa61gmuvPhdLxV,0.0,0.0,0.0
63154,2020-06-13 04:30:00,zBIq5rxdHJRwDNY,0.0,0.0,0.0
26800,2020-06-13 04:30:00,ZnxXDlPa8U1GXgE,0.0,0.0,0.0
44994,2020-06-13 04:30:00,pkci93gMrogZuBj,0.0,0.0,0.0
5619,2020-06-13 04:30:00,1IF53ai7Xc0U56Y,0.0,0.0,0.0
35905,2020-06-13 04:30:00,bvBOhCH3iADSZry,0.0,0.0,0.0



Plant 2:


,DATE_TIME,SOURCE_KEY,AC_POWER,TARGET_AC_POWER,PERSISTENCE_PREDICTION
50183,2020-06-13 01:00:00,oZZkBaNadn6DNKz,0.0,0.0,0.0
2623,2020-06-13 01:00:00,4UPUqMRk7TRMgml,0.0,0.0,0.0
12045,2020-06-13 01:00:00,Et9kgGMDl729KT4,0.0,0.0,0.0
23725,2020-06-13 01:00:00,Mx2yZCDsyf6DPfv,0.0,0.0,0.0
29145,2020-06-13 01:00:00,PeE6FRyGXUgsRhN,0.0,0.0,0.0
65089,2020-06-13 01:00:00,xoJJ8DcxJEcupym,0.0,0.0,0.0
14303,2020-06-13 01:00:00,IQ2d7wF4YD8zU1Q,0.0,0.0,0.0
56507,2020-06-13 01:00:00,rrq4fwE8jgrTyWY,0.0,0.0,0.0
43923,2020-06-13 01:00:00,mqwcsP2rE7J0TFp,0.0,0.0,0.0
53345,2020-06-13 01:00:00,q49J1IKaHRwDQnt,0.0,0.0,0.0


In [11]:
# Calculate persistence baseline metrics

def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return mae, rmse, r2


plant1_mae, plant1_rmse, plant1_r2 = calculate_metrics(
    plant1_test["TARGET_AC_POWER"],
    plant1_test["PERSISTENCE_PREDICTION"]
)

plant2_mae, plant2_rmse, plant2_r2 = calculate_metrics(
    plant2_test["TARGET_AC_POWER"],
    plant2_test["PERSISTENCE_PREDICTION"]
)

print("Plant 1 Persistence Baseline")
print("MAE :", plant1_mae)
print("RMSE:", plant1_rmse)
print("R²  :", plant1_r2)

print("\nPlant 2 Persistence Baseline")
print("MAE :", plant2_mae)
print("RMSE:", plant2_rmse)
print("R²  :", plant2_r2)

Plant 1 Persistence Baseline
MAE : 60.10463528878193
RMSE: 113.12520687156119
R²  : 0.9113377164945652

Plant 2 Persistence Baseline
MAE : 63.91827313738893
RMSE: 132.19667444076524
R²  : 0.803117093230456


In [12]:
baseline_results = pd.DataFrame({
    "Plant": ["Plant 1", "Plant 2"],
    "Model": ["Persistence", "Persistence"],
    "MAE": [plant1_mae, plant2_mae],
    "RMSE": [plant1_rmse, plant2_rmse],
    "R2": [plant1_r2, plant2_r2]
})

baseline_results

,Plant,Model,MAE,RMSE,R2
0,Plant 1,Persistence,60.104635,113.125207,0.911338
1,Plant 2,Persistence,63.918273,132.196674,0.803117


In [13]:
import os

METRICS_PATH = "../outputs/metrics/"
os.makedirs(METRICS_PATH, exist_ok=True)

baseline_results.to_csv(
    METRICS_PATH + "baseline_results.csv",
    index=False
)

print("Baseline results saved successfully.")

Baseline results saved successfully.


In [14]:
PREDICTIONS_PATH = "../outputs/predictions/"
os.makedirs(PREDICTIONS_PATH, exist_ok=True)

plant1_test[
    ["DATE_TIME", "PLANT_ID", "SOURCE_KEY",
     "AC_POWER", "TARGET_AC_POWER", "PERSISTENCE_PREDICTION"]
].to_csv(
    PREDICTIONS_PATH + "plant1_persistence_predictions.csv",
    index=False
)

plant2_test[
    ["DATE_TIME", "PLANT_ID", "SOURCE_KEY",
     "AC_POWER", "TARGET_AC_POWER", "PERSISTENCE_PREDICTION"]
].to_csv(
    PREDICTIONS_PATH + "plant2_persistence_predictions.csv",
    index=False
)

print("Persistence predictions saved successfully.")

Persistence predictions saved successfully.
